In [32]:
import os
from pathlib import Path


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "YOUR_TAVILY_API_KEY")
BASE_DIR = Path.cwd()  
DATA_DIR = BASE_DIR / "data"
DB_DIR = BASE_DIR / "db" / "cafe_db"

MENU_PATH = DATA_DIR / "cafe_menu.txt"

print("BASE_DIR :", BASE_DIR)
print("DATA_DIR :", DATA_DIR)
print("DB_DIR   :", DB_DIR)
print("MENU_PATH:", MENU_PATH)

# 폴더 생성
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DB_DIR.parent).mkdir(parents=True, exist_ok=True)


BASE_DIR : c:\mylangchain\mylangchain-app\src\mylangchain_app\000assignment
DATA_DIR : c:\mylangchain\mylangchain-app\src\mylangchain_app\000assignment\data
DB_DIR   : c:\mylangchain\mylangchain-app\src\mylangchain_app\000assignment\db\cafe_db
MENU_PATH: c:\mylangchain\mylangchain-app\src\mylangchain_app\000assignment\data\cafe_menu.txt


In [33]:
from langchain.docstore.document import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


raw_text = MENU_PATH.read_text(encoding="utf-8")
chunks = [c.strip() for c in raw_text.split("\n\n") if c.strip()]


docs = []
for chunk in chunks:
    if "메뉴:" in chunk:
        name_line = next((ln for ln in chunk.splitlines() if ln.startswith("메뉴:")), "메뉴: Unknown")
        menu_name = name_line.split("메뉴:")[-1].strip()
        docs.append(Document(page_content=chunk, metadata={"menu_name": menu_name}))

print("문서 개수:", len(docs))
assert len(docs) >= 10, "메뉴는 최소 10개 항목이어야 합니다."


emb = OpenAIEmbeddings()
vector = FAISS.from_documents(docs, emb)


DB_DIR.mkdir(parents=True, exist_ok=True)
vector.save_local(str(DB_DIR))

print("벡터 DB 생성 및 저장 완료 →", DB_DIR)


문서 개수: 10
벡터 DB 생성 및 저장 완료 → c:\mylangchain\mylangchain-app\src\mylangchain_app\000assignment\db\cafe_db


In [34]:
from typing import List
from langchain.tools import tool
from tavily import TavilyClient
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


_vector = None

def get_vector() -> FAISS:
    global _vector
    if _vector is None:
    
        _vector = FAISS.load_local(
            str(DB_DIR),
            OpenAIEmbeddings(),
            allow_dangerous_deserialization=True,
        )
    return _vector

@tool
def tavily_search_func(query: str) -> str:
    """웹에서 최신 정보 검색 (Tavily). 입력: 검색 쿼리 문자열"""
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key or api_key == "YOUR_TAVILY_API_KEY":
        return "TAVILY_API_KEY가 설정되지 않았습니다."
    client = TavilyClient(api_key=api_key)
    res = client.search(query=query, max_results=5)

    bullets = []
    for item in res.get("results", []):
        title = item.get("title", "")
        content = item.get("content", "")[:200].replace("\n", " ")
        url = item.get("url", "")
        bullets.append(f"- {title}: {content} ... ({url})")
    return "웹 검색 결과:\n" + "\n".join(bullets) if bullets else "검색 결과가 없습니다."

@tool
def wiki_summary(topic: str) -> str:
    """위키피디아에서 일반 지식 검색 및 요약. 입력: 주제 문자열"""
    wiki = WikipediaAPIWrapper(lang="ko")
    try:
        return wiki.run(topic)
    except Exception as e:
        return f"위키 검색 중 오류: {e}"

@tool
def db_search_cafe_func(query: str) -> List[str]:
    """로컬 카페 메뉴 DB에서 정보 검색. 입력: 메뉴 관련 쿼리 문자열"""
    vector = get_vector()
    hits = vector.similarity_search(query, k=3)

    outs = []
    for h in hits:
        name = h.metadata.get("menu_name", "Unknown")
        outs.append(f"[{name}]\n{h.page_content}")
    return outs


In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [tavily_search_func, wiki_summary, db_search_cafe_func]
llm_with_tools = llm.bind_tools(tools)

print("LLM에 3개 도구 바인딩 완료")


LLM에 3개 도구 바인딩 완료


In [36]:
from typing import Dict, Any

def run_cafe_agent(user_question: str) -> str:
    """
    사용자 질문 → LLM(도구 선택) → 도구 실행 → 결과 종합 → 최종 답변
    """

    ai_msg = llm_with_tools.invoke(user_question)

    tool_calls = getattr(ai_msg, "tool_calls", []) or []
    tool_results: Dict[str, Any] = {}

    for i, tc in enumerate(tool_calls):
        tool_name = tc["name"]
        args = tc.get("args", {}) or {}
        if tool_name == "db_search_cafe_func":
            res = db_search_cafe_func.invoke(args.get("query") or args.get("input") or user_question)
            tool_results["db_search_cafe_func"] = res
        elif tool_name == "tavily_search_func":
            res = tavily_search_func.invoke(args.get("query") or args.get("input") or user_question)
            tool_results["tavily_search_func"] = res
        elif tool_name == "wiki_summary":
            res = wiki_summary.invoke(args.get("topic") or args.get("input") or user_question)
            tool_results["wiki_summary"] = res


    context_lines = []
    for k, v in tool_results.items():
        if isinstance(v, list):
            v = "\n\n".join(v)
        context_lines.append(f"[{k} 결과]\n{v}")
    context_blob = "\n\n".join(context_lines) if context_lines else "도구를 사용하지 않았습니다."

    final_prompt = f"""다음은 사용자 질문과 도구 실행 결과입니다.
사용자 질문: {user_question}

도구 실행 요약:
{context_blob}

위 정보를 바탕으로, 한국어로 친절하고 간결하게 답변하세요.
가능하면 가격/재료/설명을 구조적으로 정리하고,
원문 정보에 없는 추측은 하지 마세요.
"""

    final_answer = llm.invoke(final_prompt)
    return str(final_answer.content)


In [37]:
question = "아메리카노의 가격과 특징은 무엇인가요?"
answer = run_cafe_agent(question)
print(answer)


c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\mylangchain-app-SBe-Yh6W-py3.12\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\mylangchain-app-SBe-Yh6W-py3.12\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


아메리카노에 대한 정보는 다음과 같습니다:

### 아메리카노
- **가격**: 4,500원
- **재료**: 에스프레소, 뜨거운 물
- **설명**: 원두 본연의 풍미를 깔끔하게 즐길 수 있는 기본 커피입니다.

### 디카페인 아메리카노
- **가격**: 5,000원
- **재료**: 디카페인 에스프레소, 뜨거운 물
- **설명**: 카페인 부담 없이 아메리카노의 맛을 그대로 즐길 수 있습니다.

아메리카노는 에스프레소를 뜨거운 물로 희석하여 마시는 커피 음료로, 일반적인 드립 커피와 비슷한 농도를 가지지만 풍미는 다릅니다.
